In [35]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

chat = ChatGroq(
    temperature=2,
    model="llama3-8b-8192",
    # api_key="" # Optional if not set as an environment variable
)

system = "You are a helpful assistant."
human = "{text}"
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", human),
    ]
)

chain = prompt | chat
chain.invoke(
    {
        "text": "Explain the importance of low latency for LLMs.",
    }
)

AIMessage(content="Low latency is crucial for Large Language Models (LLMs) as it directly impacts their overall performance, user experience, and potential applications. Here are some reasons why low latency is important:\n\n1. **Reduced Response Time**: With low latency, users receive rapid intelligent responses from LLMs, which becomes conversational in nature. Quick turnaround times lead to a sense of fluidity and efficiency, indicating the AI assistant understands their intent efficiently.\n\n2. **Conversation Perceptinble Scale**: In real-world applications, we may likely see concurrent requests/interations; To processes a high quantity, Low complexity, is key efficiency factor on backfill resources better more time spend from users perspective and therefore becomes scale-issue for efficiency wise\n\nA possible alternative focus & mitigation, focusing implementation & resource scalability - more of distributional parallel efforts (shared- hosting) Over-provisioning model scalabili

In [36]:
from typing import Optional

from langchain_core.tools import tool


@tool
def get_current_weather(location: str, unit: Optional[str]):
    """Get the current weather in a given location"""
    return "Cloudy with a chance of rain."


tool_model = chat.bind_tools(
    [get_current_weather],
    tool_choice="auto",
)

res = tool_model.invoke("What is the weather like in San Francisco and Tokyo?")

res.tool_calls

[]

In [37]:
from langchain_core.pydantic_v1 import BaseModel, Field


class Joke(BaseModel):
    """Joke to tell user."""

    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline to the joke")
    rating: Optional[int] = Field(description="How funny the joke is, from 1 to 10")


structured_llm = chat.with_structured_output(Joke)

structured_llm.invoke("Tell me a joke about cats")

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<tool-use>{"\n{id: pending,\n  type: method,\n  "function": {\n   name: Joke},\nfunctions!{[{"parameters",\n    setup: WHAT\\\nhave we\ndo we cat before bedtime?",}]?}}}</software-link-unit>'}}

In [ ]:
llm = ChatGroq(model="llama3-8b-8192")

In [ ]:
json_schema = {
    "title": "joke",
    "description": "Joke to tell user.",
    "type": "object",
    "properties": {
        "setup": {
            "type": "string",
            "description": "The setup of the joke",
        },
        "punchline": {
            "type": "string",
            "description": "The punchline to the joke",
        },
        "rating": {
            "type": "integer",
            "description": "How funny the joke is, from 1 to 10",
        },
    },
    "required": [
        "setup",
        "punchline",
    ],
}
structured_llm = llm.with_structured_output(json_schema)

structured_llm.invoke("Tell me a joke about cats")

{'setup': 'Why did the cat join a band?',
 'punchline': 'Because it wanted to be the purr-cussionist!',
 'rating': 8}

In [ ]:
from typing import Union


class ConversationalResponse(BaseModel):
    """Respond in a conversational manner. Be kind and helpful."""

    response: str = Field(description="A conversational response to the user's query")


class Response(BaseModel):
    output: Union[
        Joke,
        ConversationalResponse,
    ]


structured_llm = llm.with_structured_output(Response)

structured_llm.invoke("Tell me a joke about cats")

Response(output=Joke(setup='Why did the cat join a band?', punchline='Why did the cat join a band? Because it wanted to be the purr-cussionist!', rating=8))

In [ ]:
structured_llm.invoke("How are you today?")

Response(output=ConversationalResponse(response="I'm doing well, thank you for asking! How about you?"))

In [ ]:
structured_llm = llm.with_structured_output(json_schema)

for chunk in structured_llm.stream("Tell me a joke about cats"):
    print(chunk)

{'setup': 'Why did the cat join a band?', 'punchline': 'Because it wanted to be the purr-cussionist!', 'rating': 7}


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system = """You are a hilarious comedian. Your specialty is knock-knock jokes. \
Return a joke which has the setup (the response to "Who's there?") and the final punchline (the response to "<setup> who?").

Here are some examples of jokes:

example_user: Tell me a joke about planes
example_assistant: {{"setup": "Why don't planes ever get tired?", "punchline": "Because they have rest wings!", "rating": 2}}

example_user: Tell me another joke about planes
example_assistant: {{"setup": "Cargo", "punchline": "Cargo 'vroom vroom', but planes go 'zoom zoom'!", "rating": 10}}

example_user: Now about caterpillars
example_assistant: {{"setup": "Caterpillar", "punchline": "Caterpillar really slow, but watch me turn into a butterfly and steal the show!", "rating": 5}}"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{input}"),
    ]
)

few_shot_structured_llm = prompt | structured_llm
few_shot_structured_llm.invoke("what's something funny about woodpeckers")

{'setup': 'Why did the woodpecker go to the doctor?',
 'punchline': 'Because it had a lot of hang-ups!',
 'rating': 8}

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

examples = [
    HumanMessage(
        "Tell me a joke about planes",
        name="example_user",
    ),
    AIMessage(
        "",
        name="example_assistant",
        tool_calls=[
            {
                "name": "joke",
                "args": {
                    "setup": "Why don't planes ever get tired?",
                    "punchline": "Because they have rest wings!",
                    "rating": 2,
                },
                "id": "1",
            }
        ],
    ),
    # Most tool-calling models expect a ToolMessage(s) to follow an AIMessage with tool calls.
    ToolMessage(
        "",
        tool_call_id="1",
    ),
    # Some models also expect an AIMessage to follow any ToolMessages,
    # so you may need to add an AIMessage here.
    HumanMessage(
        "Tell me another joke about planes",
        name="example_user",
    ),
    AIMessage(
        "",
        name="example_assistant",
        tool_calls=[
            {
                "name": "joke",
                "args": {
                    "setup": "Cargo",
                    "punchline": "Cargo 'vroom vroom', but planes go 'zoom zoom'!",
                    "rating": 10,
                },
                "id": "2",
            }
        ],
    ),
    ToolMessage(
        "",
        tool_call_id="2",
    ),
    HumanMessage(
        "Now about caterpillars",
        name="example_user",
    ),
    AIMessage(
        "",
        tool_calls=[
            {
                "name": "joke",
                "args": {
                    "setup": "Caterpillar",
                    "punchline": "Caterpillar really slow, but watch me turn into a butterfly and steal the show!",
                    "rating": 5,
                },
                "id": "3",
            }
        ],
    ),
    ToolMessage(
        "",
        tool_call_id="3",
    ),
]
system = """You are a hilarious comedian. Your specialty is knock-knock jokes. \
Return a joke which has the setup (the response to "Who's there?") \
and the final punchline (the response to "<setup> who?")."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("placeholder", "{examples}"),
        ("human" "{input}"),
    ]
)
few_shot_structured_llm = prompt | structured_llm
few_shot_structured_llm.invoke(
    {
        "input": "crocodiles",
        "examples": examples,
    }
)

{'setup': 'Why did the human-crocodile go to the party?',
 'punchline': 'Because it was a snappy dresser!',
 'rating': 3}

In [ ]:
structured_llm = llm.with_structured_output(Joke, method="json_mode")

structured_llm.invoke(
    "Tell me a joke about cats, respond in JSON with `setup` and `punchline` keys"
)

Joke(setup='Why did the cat join a band?', punchline='Because it wanted to be the purr-cussionist!', rating=None)

In [ ]:
structured_llm = llm.with_structured_output(Joke, include_raw=True)

structured_llm.invoke(
    "Tell me a joke about cats, respond in JSON with `setup` and `punchline` keys"
)

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_rawd', 'function': {'arguments': '{"punchline":"Why did the cat join a band? Because it wanted to be the purr-cussionist!","setup":"Why did the cat decide to pursue a career in music?"}', 'name': 'Joke'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 903, 'total_tokens': 1006, 'completion_time': 0.081372658, 'prompt_time': 0.140863872, 'queue_time': None, 'total_time': 0.22223653}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_873a560973', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-b5866855-52e0-43c3-a1e0-108363610742-0', tool_calls=[{'name': 'Joke', 'args': {'punchline': 'Why did the cat join a band? Because it wanted to be the purr-cussionist!', 'setup': 'Why did the cat decide to pursue a career in music?'}, 'id': 'call_rawd', 'type': 'tool_call'}], usage_metadata={'input_tokens': 903, 'output_tokens': 103, 'total_tokens': 100

In [ ]:
from typing import List

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field


class Person(BaseModel):
    """Information about a person."""

    name: str = Field(..., description="The name of the person")
    height_in_meters: float = Field(
        ..., description="The height of the person expressed in meters."
    )


class People(BaseModel):
    """Identifying information about all people in a text."""

    people: List[Person]


# Set up a parser
parser = PydanticOutputParser(pydantic_object=People)

# Prompt
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the user query. Wrap the output in `json` tags\n{format_instructions}",
        ),
        ("human", "{query}"),
    ]
).partial(
    format_instructions=parser.get_format_instructions(),
)

In [ ]:
query = "Anna is 23 years old and she is 6 feet tall"

print(prompt.invoke(query).to_string())

System: Answer the user query. Wrap the output in `json` tags
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Identifying information about all people in a text.", "properties": {"people": {"title": "People", "type": "array", "items": {"$ref": "#/definitions/Person"}}}, "required": ["people"], "definitions": {"Person": {"title": "Person", "description": "Information about a person.", "type": "object", "properties": {"name": {"title": "Name", "description": "The name of the person", "type": "string"}, "height_in_meters": {"title": "Height In Meters", "description": "The heig

In [ ]:
chain = prompt | llm | parser

chain.invoke(
    {
        "query": query,
    }
)

People(people=[Person(name='Anna', height_in_meters=1.8288)])

In [ ]:
import json
import re
from typing import List

from langchain_core.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field


class Person(BaseModel):
    """Information about a person."""

    name: str = Field(..., description="The name of the person")
    height_in_meters: float = Field(
        ..., description="The height of the person expressed in meters."
    )


class People(BaseModel):
    """Identifying information about all people in a text."""

    people: List[Person]


# Prompt
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the user query. Output your answer as JSON that  "
            "matches the given schema: ```json\n{schema}\n```. "
            "Make sure to wrap the answer in ```json and ``` tags",
        ),
        ("human", "{query}"),
    ]
).partial(
    schema=People.schema(),
)


# Custom parser
def extract_json(message: AIMessage) -> List[dict]:
    """Extracts JSON content from a string where JSON is embedded between ```json and ``` tags.

    Parameters:
        text (str): The text containing the JSON content.

    Returns:
        list: A list of extracted JSON strings.
    """
    text = message.content
    # Define the regular expression pattern to match JSON blocks
    pattern = r"```json(.*?)```"

    # Find all non-overlapping matches of the pattern in the string
    matches = re.findall(
        pattern,
        text,
        re.DOTALL,
    )

    # Return the list of matched JSON strings, stripping any leading or trailing whitespace
    try:
        return [json.loads(match.strip()) for match in matches]
    except Exception:
        raise ValueError(f"Failed to parse: {message}")

In [ ]:
query = "Anna is 23 years old and she is 6 feet tall"

print(prompt.format_prompt(query=query).to_string())

System: Answer the user query. Output your answer as JSON that  matches the given schema: ```json
{'title': 'People', 'description': 'Identifying information about all people in a text.', 'type': 'object', 'properties': {'people': {'title': 'People', 'type': 'array', 'items': {'$ref': '#/definitions/Person'}}}, 'required': ['people'], 'definitions': {'Person': {'title': 'Person', 'description': 'Information about a person.', 'type': 'object', 'properties': {'name': {'title': 'Name', 'description': 'The name of the person', 'type': 'string'}, 'height_in_meters': {'title': 'Height In Meters', 'description': 'The height of the person expressed in meters.', 'type': 'number'}}, 'required': ['name', 'height_in_meters']}}}
```. Make sure to wrap the answer in ```json and ``` tags
Human: Anna is 23 years old and she is 6 feet tall


In [ ]:
chain = prompt | llm | extract_json

chain.invoke(
    {
        "query": query,
    }
)

[{'people': [{'name': 'Anna', 'height_in_meters': 1.8288}]}]

In [ ]:
chat = ChatGroq(
    temperature=2,
    model="llama3-8b-8192",
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "human",
            "Write a Limerick about {topic}",
        ),
    ]
)
chain = prompt | chat
await chain.ainvoke(
    {
        "topic": "The Sun",
    }
)

AIMessage(content='There once was a sun in the sky,\nWhose warmth and bright light did not die,\nIt shone with great might,\nAnd lit up the night,\nAnd brought joy to all who did fly.', response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 18, 'total_tokens': 60, 'completion_time': 0.032621843, 'prompt_time': 0.004065488, 'queue_time': None, 'total_time': 0.036687331}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_873a560973', 'finish_reason': 'stop', 'logprobs': None}, id='run-478d2ceb-6760-486b-b29f-c7bb27cc4903-0', usage_metadata={'input_tokens': 18, 'output_tokens': 42, 'total_tokens': 60})

In [ ]:
chat = ChatGroq(
    temperature=2,
    model="llama3-8b-8192",
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "human",
            "Write a haiku about {topic}",
        )
    ]
)
chain = prompt | chat
for chunk in chain.stream(
    {
        "topic": "The Moon",
    }
):
    print(
        chunk.content,
        end="",
        flush=True,
    )

Silvery glow bright
Moon's gentle light on my face
Peaceful nightvery glow bright
Moon's gentle light on my face
Peaceful night's delight

In [ ]:
chat = ChatGroq(
    model="llama3-70b-8192",
    model_kwargs={
        "response_format": {
            "type": "json_object",
        }
    },
)

system = """
You are a helpful assistant.
Always respond with a JSON object with two string keys: "response" and "followup_question".
"""
human = "{question}"
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", human),
    ]
)

chain = prompt | chat

chain.invoke(
    {
        "question": "what bear is best?",
    }
)

AIMessage(content='{"response": "That\'s a tough one! But if I had to pick, I\'d say the giant panda is pretty amazing. They\'re so cute and gentle, and they love to munch on bamboo!", "followup_question": "What do you think is the most interesting thing about giant pandas?"}', response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 50, 'total_tokens': 113, 'completion_time': 0.192442719, 'prompt_time': 0.012093368, 'queue_time': None, 'total_time': 0.204536087}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_753a4aecf6', 'finish_reason': 'stop', 'logprobs': None}, id='run-ca22d0f9-30d5-468f-a84d-36a9208a6cac-0', usage_metadata={'input_tokens': 50, 'output_tokens': 63, 'total_tokens': 113})